<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:46px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">External Validation · Domain Shift · Trustworthy Deep Learning</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Akciğer-Odaklı ViT — Dış (Bağımsız Kaynak) Doğrulama</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Eğitimde hiç görülmemiş, farklı merkez/cihaz/popülasyondan veri kümeleri üzerinde dağıtım kayması (domain shift) altında başarımın ölçülmesi</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Model:</b> <code>lung_focused_vit_pneumonia.pth</code> (kendi eğittiğiniz)</div>
    <div><b>Ön-işleme:</b> ianpan akciğer maskesi + RoI (checkpoint'ten birebir)</div>
    <div><b>Dış set 1:</b> RSNA Pneumonia Detection Challenge (yetişkin)</div>
    <div><b>Dış set 2:</b> NIH ChestX-ray14 (yetişkin)</div>
    <div><b>Birincil metrik:</b> ROC-AUC (eşikten bağımsız)</div>
    <div><b>Ek:</b> Kalibrasyon (Brier, ECE), duyarlılık/özgüllük</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Amaç:</b> Eğitim/test verisi (Kermany pediatrik, Guangzhou) ile aynı dağıtımdaki yüksek skor, modelin <i>başka bir hastanenin cihazında</i> çalışacağını kanıtlamaz. Bu notebook, modeli eğitiminden tamamen bağımsız kaynaklardan gelen setlerde sınar — kısayolsuzluğun ve genellemenin asıl sınavı budur.
  </div>
</div>

## 1. Hangi Veri Setleri ve Neden? <a id='1'></a>

Modeliniz **Kermany / Guangzhou pediatrik** setinde (1–5 yaş, tek merkez) eğitildi. Geçerli bir dış doğrulama seti, bu dağıtımla **örtüşmemelidir**. Aşağıdaki iki set bu koşulu sağlar; ikisi de NIH/RSNA kaynaklı, **yetişkin** popülasyondur.

| Set | Kaynak | Format | Etiket mantığı | Rol |
|-----|--------|--------|----------------|-----|
| **RSNA Pneumonia Detection Challenge** | NIH + RSNA (ABD, yetişkin) | DICOM | `Lung Opacity` = pnömoni, `Normal` = sağlıklı (radyolog gözden geçirmeli, ikili) | **Birincil / titiz** |
| **NIH ChestX-ray14** | NIH (ABD, yetişkin) | PNG | Etikette `Pneumonia` = pnömoni, `No Finding` = sağlıklı (rapor-NLP etiketi) | İkincil / kolay format |

**Neden bu ikisi?** Her ikisi de eğitim setinizden bağımsız kaynaktır; aralarındaki performans farkı doğrudan **dağıtım kaymasını** ölçer (RSNA aslında NIH'in bir alt kümesidir, ama etiket kaliteleri ve formatları farklıdır).

**Neden COVID-19 Radiography DB kullanılmadı?** O setin "Viral Pneumonia" ve erken "Normal" görüntüleri **tam da sizin eğitim setinizin kaynağı olan Kermany/Guangzhou pediatrik setinden** alınmıştır. Yani dış doğrulama gibi görünür ama eğitim dağıtımıyla örtüşür (veri sızıntısı) — bu yüzden kasıtlı olarak dışlandı.

### Kaggle'da kurulum (Add Input)
Sağ üstten **Add Input** ile şunları ekleyin (en az biri yeterli; ikisi birlikte daha güçlü):
- **RSNA**: "Competitions" sekmesinde **RSNA Pneumonia Detection Challenge** (`rsna-pneumonia-detection-challenge`).
- **NIH**: "Datasets" sekmesinde **NIH Chest X-rays** (`nih-chest-xrays/data`).
- **Modeliniz**: önceki notebook'ta oluşturduğunuz **`lung-focused-vit-pneumonia`** dataset'i (içinde `lung_focused_vit_pneumonia.pth`).

> **İnternet açık olmalı** (Settings → Internet → On): `ianpan/chest-x-ray-basic` segmentasyon modeli HuggingFace'ten indirilir.
> Bu notebook eklenen setleri ve `.pth` dosyasını `/kaggle/input/` altında **otomatik bulur**; yol elle girmenize gerek yoktur.

## 2. Kurulum, İçe Aktarmalar ve Çalıştırma Ayarları <a id='2'></a>

`MAX_PER_CLASS`, her dış setten sınıf başına kaç görüntü değerlendirileceğini belirler. Segmentasyon her görüntü için bir ileri-geçiş gerektirdiğinden, tüm seti (10⁴–10⁵ görüntü) işlemek bir Kaggle oturumunda pratik değildir. Varsayılan **400/sınıf** (≈800 görüntü/set), AUC için istatistiksel olarak yeterli ve dengeli bir örneklemdir; GPU ile birkaç dakikada biter. Daha sıkı bir tahmin için artırabilirsiniz.

In [ ]:
import os, sys, glob, json, random, warnings, types, subprocess
warnings.filterwarnings('ignore')

# pydicom (RSNA DICOM icin) — Kaggle'da genelde kuruludur; yoksa kur
try:
    import pydicom
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydicom"])
    import pydicom

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score,
                             f1_score, brier_score_loss)

# ----------------- ÇALIŞTIRMA AYARLARI -----------------
MAX_PER_CLASS = 400      # her dis setten sinif basina goruntu (hiz/dogruluk dengesi)
DECISION_THR  = 0.5      # ikili karar esigi (P(pnomoni) >= thr -> PNEUMONIA)
SEED          = 42
# Kendi ic-test skorunuzu kiyas grafigine eklemek isterseniz doldurun (opsiyonel):
INTERNAL_REF  = {"name": "Ic Test (Kermany)", "auc": None, "f1": None}
# -------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.02)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

print(f"Cihaz: {device} | PyTorch {torch.__version__}")
print(f"MAX_PER_CLASS = {MAX_PER_CLASS}, karar esigi = {DECISION_THR}")

## 3. Girdilerin Otomatik Keşfi <a id='3'></a>

`/kaggle/input/` altında üç şey aranır: (1) model checkpoint'i (`*.pth`), (2) RSNA işaretçi dosyası (`stage_2_detailed_class_info.csv`), (3) NIH işaretçi dosyası (`Data_Entry_2017.csv`). Bulunanlar değerlendirmeye alınır, bulunamayan set atlanır.

In [ ]:
INPUT = "/kaggle/input"

def find_dir_with(root, fname):
    """fname adli dosyayi iceren ilk dizini dondurur (yoksa None)."""
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

# --- Model checkpoint ---
pths = glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True)
CKPT_PATH = None
for p in pths:
    if "lung_focused" in os.path.basename(p).lower():
        CKPT_PATH = p; break
if CKPT_PATH is None and pths:
    CKPT_PATH = pths[0]

# --- Dis setler ---
RSNA_BASE = find_dir_with(INPUT, "stage_2_detailed_class_info.csv")
NIH_BASE  = find_dir_with(INPUT, "Data_Entry_2017.csv")

print("Bulunanlar:")
print(f"  Model (.pth) : {CKPT_PATH}")
print(f"  RSNA tabani  : {RSNA_BASE}")
print(f"  NIH tabani   : {NIH_BASE}")
assert CKPT_PATH is not None, ("Model .pth bulunamadi! 'lung-focused-vit-pneumonia' "
                               "dataset'ini Add Input ile ekleyin.")
if RSNA_BASE is None and NIH_BASE is None:
    raise RuntimeError("Hicbir dis set bulunamadi! RSNA ve/veya NIH'i Add Input ile ekleyin.")

## 4. Modeli Yükle (Checkpoint'ten) <a id='4'></a>

Checkpoint yalnızca ağırlıkları değil, eğitimde kullanılan **tüm ön-işleme parametrelerini** (`CONFIG`) ve sınıf eşlemesini taşır. Bunları buradan okuyarak doğrulama hattının eğitimle **birebir** aynı olmasını garanti ederiz. ViT mimarisi `weights=None` ile kurulur (ImageNet ağırlığı indirilmez); ağırlıklar tamamen sizin checkpoint'inizden gelir.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
CFG          = ckpt["config"]
CLASS_NAMES  = ckpt["class_names"]          # ['NORMAL','PNEUMONIA']
CLASS_TO_IDX = ckpt["class_to_idx"]         # {'NORMAL':0,'PNEUMONIA':1}
PNEU_IDX     = CLASS_TO_IDX["PNEUMONIA"]    # pozitif sinif indeksi (=1)

def build_vit_inference(num_classes, dropout):
    m = models.vit_b_16(weights=None)       # mimari yapi; agirlik indirilmez
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

model = build_vit_inference(ckpt["num_classes"], CFG.get("dropout", 0.1)).to(device)
missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

print("Model yuklendi.")
print(f"  Mimari        : {ckpt.get('arch')}")
print(f"  Egitim ValF1  : {ckpt.get('best_val_f1')}")
print(f"  Siniflar      : {CLASS_TO_IDX}")
print(f"  Girdi boyutu  : {CFG['img_size']} | mean={CFG['mean'][0]:.4f} std={CFG['std'][0]:.4f}")
print(f"  Maske payi    : dilate={CFG['mask_dilate_frac']} roi_pad={CFG['roi_pad_frac']} feather={CFG['mask_feather']} fill={CFG['fill_mode']}")

## 5. Anatomik Segmentasyon Modeli (ianpan) <a id='5'></a>

Eğitimle **aynı** segmentasyon modeli (`ianpan/chest-x-ray-basic`) yüklenir. Aynı maske → aynı RoI → aynı girdi dağılımı. İnternet açık olmalıdır.

In [ ]:
import transformers
from transformers import AutoModel

print("ianpan/chest-x-ray-basic yukleniyor... (transformers", transformers.__version__, ")")

def _load_seg():
    return AutoModel.from_pretrained(
        "ianpan/chest-x-ray-basic", trust_remote_code=True
    ).to(device).eval()

try:
    seg_model = _load_seg()
except Exception as e:
    # Surum uyumsuzlugu: 'all_tied_weights_keys' eksikligi -> bos varsayilan ekleyip tekrar dene
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise

print("Segmentasyon modeli hazir.")

## 6. Ön-İşleme Hattı — Eğitimle Birebir Aynı <a id='6'></a>

Aşağıdaki fonksiyonlar eğitim notebook'undakilerle **aynıdır**; tek fark, DICOM (RSNA) dosyalarını da okuyabilen genişletilmiş bir yükleyicidir. Bir görüntü için akış: **DICOM/PNG oku → ianpan maskesi (kalp hariç) → genişlet + yumuşat + mean-dolgu → RoI kırp → 224 → Normalize → P(pnömoni)**.

In [ ]:
def read_dicom_u8(path):
    """RSNA DICOM -> (H,W) uint8 gri. MONOCHROME1 ise ters cevirir."""
    dcm = pydicom.dcmread(path)
    arr = dcm.pixel_array.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = 1.0 - arr
    return (arr * 255).astype(np.uint8)


def load_image_any(path, short_max):
    """PNG/JPG/DICOM -> (rgb_u8, gray_u8). Kisa kenari short_max'a indirir."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        pil = Image.fromarray(read_dicom_u8(path)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0*s)), int(round(H0*s))), Image.BILINEAR)
    rgb  = np.asarray(pil).astype(np.uint8)
    gray = np.asarray(pil.convert("L"))
    return rgb, gray


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    """ianpan ile sag+sol akciger maskesi (kalp haric)."""
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)   # 3=kalp DISLANIR


def make_lung_focused(rgb_u8, lung_u8, cfg):
    """Maske + RoI kirp + 224. Doner: (img224_u8, mask224_u8, used_bool). Egitimle ayni."""
    H, W = lung_u8.shape
    short = min(H, W); S = cfg["img_size"]
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        return fb, np.ones((S, S), np.uint8), False
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*dil+1, 2*dil+1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0: f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]
    if cfg["fill_mode"] == "mean":
        fill = np.array([m*255.0 for m in cfg["mean"]], dtype=np.float32)
    else:
        fill = np.zeros(3, dtype=np.float32)
    masked = rgb_u8.astype(np.float32) * soft + fill[None, None, :] * (1.0 - soft)
    masked = masked.clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max()); x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0-pad); x0 = max(0, x0-pad)
    y1 = min(H-1, y1+pad); x1 = min(W-1, x1+pad)
    bh, bw = (y1-y0+1), (x1-x0+1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0+y1)//2, (x0+x1)//2
    ty0 = max(0, cy-side//2); tx0 = max(0, cx-side//2)
    ty1 = min(H, ty0+side);   tx1 = min(W, tx0+side)
    ty0 = max(0, ty1-side);   tx0 = max(0, tx1-side)
    roi = masked[ty0:ty1, tx0:tx1]
    img224 = cv2.resize(roi, (S, S), interpolation=cv2.INTER_AREA)
    return img224, None, True


eval_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["mean"], CFG["std"]),
])

@torch.inference_mode()
def infer_path(path):
    """Tek goruntu -> (P(pnomoni), used_bool)."""
    rgb, gray = load_image_any(path, CFG["orig_short_max"])
    lung = lung_mask_ianpan(gray, gray.shape)
    img224, _, used = make_lung_focused(rgb, lung, CFG)
    x = eval_tf(Image.fromarray(img224)).unsqueeze(0).to(device)
    p = torch.softmax(model(x), 1)[0, PNEU_IDX].item()
    return p, used

print("On-isleme hatti hazir.")

## 7. Dış Setlerin Etiket Tabloları <a id='7'></a>

Her set için dengeli (sınıf başına `MAX_PER_CLASS`) bir örneklem kurulur.

- **RSNA:** `Lung Opacity` → PNEUMONIA, `Normal` → NORMAL. Belirsiz `No Lung Opacity / Not Normal` sınıfı **dışlanır** (pnömoni değil ama sağlıklı da değil — ikili teste gürültü katar).
- **NIH:** Etiketinde `Pneumonia` geçenler → PNEUMONIA, `No Finding` → NORMAL. Diğer tüm bulgular dışlanır. (NIH etiketleri rapordan NLP ile çıkarıldığı için ~%90 doğruluktadır; bu gürültü sonuç yorumunda dikkate alınmalıdır.)

In [ ]:
def build_rsna_items(base, k, seed=SEED):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(os.path.join(img_dir, p + ".dcm"), PNEU_IDX)       for p in pos[:k]]
    items += [(os.path.join(img_dir, p + ".dcm"), 1 - PNEU_IDX)   for p in neg[:k]]
    return items

def build_nih_items(base, k, seed=SEED):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    labels = df["Finding Labels"].astype(str)
    is_pneu   = labels.apply(lambda s: "Pneumonia" in s.split("|"))
    is_normal = labels.apply(lambda s: s.strip() == "No Finding")
    # tum PNG'lerin yol indeksi (dosya adi -> tam yol)
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    def paths_for(mask_):
        out = []
        for name in df[mask_]["Image Index"].tolist():
            p = index.get(name)
            if p: out.append(p)
        return out
    pos, neg = paths_for(is_pneu), paths_for(is_normal)
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(p, PNEU_IDX)     for p in pos[:k]]
    items += [(p, 1 - PNEU_IDX) for p in neg[:k]]
    return items

DATASETS = []   # (isim, items)
if RSNA_BASE is not None:
    rsna = build_rsna_items(RSNA_BASE, MAX_PER_CLASS)
    DATASETS.append(("RSNA (yetiskin)", rsna))
    print(f"RSNA : {len(rsna)} goruntu ({sum(l==PNEU_IDX for _,l in rsna)} pnomoni / {sum(l!=PNEU_IDX for _,l in rsna)} normal)")
if NIH_BASE is not None:
    nih = build_nih_items(NIH_BASE, MAX_PER_CLASS)
    DATASETS.append(("NIH ChestX-ray14 (yetiskin)", nih))
    print(f"NIH  : {len(nih)} goruntu ({sum(l==PNEU_IDX for _,l in nih)} pnomoni / {sum(l!=PNEU_IDX for _,l in nih)} normal)")
print(f"\nDegerlendirilecek set sayisi: {len(DATASETS)}")

## 8. Çıkarım (Inference) <a id='8'></a>

Her set için tüm görüntüler ön-işleme hattından geçirilip P(pnömoni) toplanır. Segmentasyonun boş maske döndürdüğü (fallback) görüntüler ayrıca sayılır; yüksek fallback oranı, modelin o sette anatomiyi tanımakta zorlandığını (ek bir dağıtım-kayması işareti) gösterir.

In [ ]:
def run_inference(items, name):
    y_true, y_prob, used_flags = [], [], []
    t0 = __import__("time").time()
    for i, (path, label) in enumerate(items):
        try:
            p, used = infer_path(path)
        except Exception as e:
            # bozuk/okunamayan dosyayi atla
            continue
        y_true.append(label); y_prob.append(p); used_flags.append(used)
        if (i + 1) % 100 == 0:
            print(f"  [{name}] {i+1}/{len(items)} ...")
    dt = __import__("time").time() - t0
    yt = np.array(y_true); yp = np.array(y_prob); uf = np.array(used_flags)
    fb = (~uf).mean() * 100 if len(uf) else 0.0
    print(f"  [{name}] bitti: {len(yt)} goruntu, {dt:.0f} sn, fallback %{fb:.1f}")
    return yt, yp, uf

RESULTS = {}
for name, items in DATASETS:
    print(f"\n>>> {name} cikarimi basliyor ({len(items)} goruntu)...")
    yt, yp, uf = run_inference(items, name)
    RESULTS[name] = {"y_true": yt, "y_prob": yp, "used": uf}

## 9. Set Bazında Metrikler ve Grafikler <a id='9'></a>

Her set için: karışıklık matrisi (eşik=`DECISION_THR`), ROC eğrisi (**AUC** — eşikten bağımsız, kıyas için birincil metrik), Precision-Recall eğrisi (**AP**) ve **kalibrasyon** diyagramı (Brier + ECE). Ayrıca tanı amaçlı, o sette F1'i en yükselten eşik (oracle) da raporlanır — bu eşiğin gerçek dağıtımda bilinemeyeceği unutulmamalıdır.

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece, N = 0.0, len(y_true)
    for i in range(n_bins):
        m = (y_prob > edges[i]) & (y_prob <= edges[i+1])
        if m.sum() == 0: continue
        conf = y_prob[m].mean()
        acc  = y_true[m].mean()
        ece += (m.sum() / N) * abs(acc - conf)
    return ece

def best_f1_threshold(y_true, y_prob):
    ts = np.linspace(0.05, 0.95, 19)
    f1s = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in ts]
    j = int(np.argmax(f1s))
    return ts[j], f1s[j]

def evaluate(name, y_true, y_prob, thr=DECISION_THR):
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-9)          # duyarlilik (recall, pnomoni)
    spec = tn / (tn + fp + 1e-9)          # ozgulluk (normal)
    prec = tp / (tp + fp + 1e-9)
    f1   = 2 * prec * sens / (prec + sens + 1e-9)
    acc  = (y_pred == y_true).mean()
    fpr, tpr, _ = roc_curve(y_true, y_prob); roc_auc = auc(fpr, tpr)
    pr, rc, _   = precision_recall_curve(y_true, y_prob); ap = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob); ece = expected_calibration_error(y_true, y_prob)
    bt, bf1 = best_f1_threshold(y_true, y_prob)

    fig, ax = plt.subplots(1, 4, figsize=(20, 4.6))
    fig.suptitle(f"{name}  —  AUC={roc_auc:.3f} | AP={ap:.3f} | Acc@{thr:.2f}={acc:.3f} | F1@{thr:.2f}={f1:.3f}",
                 fontsize=12, fontweight='bold')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax[0],
                xticklabels=['NORMAL', 'PNEU'], yticklabels=['NORMAL', 'PNEU'])
    ax[0].set_title(f'Karisiklik (esik={thr:.2f})'); ax[0].set_xlabel('Tahmin'); ax[0].set_ylabel('Gercek')
    ax[1].plot(fpr, tpr, color='#1f77b4', lw=2, label=f'AUC={roc_auc:.3f}')
    ax[1].plot([0, 1], [0, 1], 'k--', lw=1); ax[1].set_title('ROC'); ax[1].set_xlabel('1-Ozgulluk'); ax[1].set_ylabel('Duyarlilik'); ax[1].legend(loc='lower right')
    ax[2].plot(rc, pr, color='#2ca02c', lw=2, label=f'AP={ap:.3f}')
    ax[2].set_title('Precision-Recall'); ax[2].set_xlabel('Duyarlilik'); ax[2].set_ylabel('Kesinlik'); ax[2].legend(loc='lower left')
    bins = np.linspace(0, 1, 11); idx = np.digitize(y_prob, bins) - 1
    bx, by = [], []
    for b in range(10):
        m = idx == b
        if m.sum() > 0:
            bx.append(y_prob[m].mean()); by.append(y_true[m].mean())
    ax[3].plot([0, 1], [0, 1], 'k--', lw=1, label='Ideal')
    ax[3].plot(bx, by, 'o-', color='#d62728', lw=2, label=f'Brier={brier:.3f}\nECE={ece:.3f}')
    ax[3].set_title('Kalibrasyon'); ax[3].set_xlabel('Tahmin olasilik'); ax[3].set_ylabel('Gercek pnomoni orani'); ax[3].legend(loc='upper left')
    plt.tight_layout(); plt.savefig(f"ext_{name.split()[0].lower()}.png", dpi=130, bbox_inches='tight'); plt.show()

    print(f"  [{name}] Duyarlilik={sens:.3f} Ozgulluk={spec:.3f} Kesinlik={prec:.3f} | "
          f"oracle esik={bt:.2f} -> F1={bf1:.3f}")
    return {"set": name, "N": len(y_true), "AUC": roc_auc, "AP": ap,
            "Acc@thr": acc, "F1@thr": f1, "Duyarlilik": sens, "Ozgulluk": spec,
            "Brier": brier, "ECE": ece, "OracleThr": bt, "OracleF1": bf1}

SUMMARY = []
for name, r in RESULTS.items():
    SUMMARY.append(evaluate(name, r["y_true"], r["y_prob"]))

## 10. Setler Arası Özet ve Kıyas <a id='10'></a>

Birincil kıyas metriği **AUC**'dir; eşikten bağımsız olduğu için farklı prevalans/etiket tanımları arasında en adil karşılaştırmayı sağlar. İç-test skorunuzu yukarıdaki `INTERNAL_REF` sözlüğüne girdiyseniz, grafikte kesikli çizgi olarak gösterilir.

In [ ]:
df_sum = pd.DataFrame(SUMMARY).set_index("set")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("="*78)
print("  DIS DOGRULAMA OZETI")
print("="*78)
print(df_sum[["N", "AUC", "AP", "Acc@thr", "F1@thr", "Duyarlilik", "Ozgulluk", "Brier", "ECE"]].to_string())
print("="*78)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Dis Doğrulama — Setler Arası Karşılaştırma", fontsize=13, fontweight='bold')
xs = np.arange(len(df_sum))
ax[0].bar(xs, df_sum["AUC"], color='#1f77b4', width=0.55)
for i, v in enumerate(df_sum["AUC"]): ax[0].text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=10)
if INTERNAL_REF.get("auc"):
    ax[0].axhline(INTERNAL_REF["auc"], color='purple', ls='--', lw=1.5, label=f"{INTERNAL_REF['name']} AUC={INTERNAL_REF['auc']:.3f}")
    ax[0].legend()
ax[0].set_xticks(xs); ax[0].set_xticklabels([s.split()[0] for s in df_sum.index]); ax[0].set_ylim(0.4, 1.0); ax[0].set_title("ROC-AUC (birincil metrik)"); ax[0].axhline(0.5, color='grey', ls=':', lw=1)
w = 0.35
ax[1].bar(xs - w/2, df_sum["Duyarlilik"], width=w, label='Duyarlilik', color='#d62728')
ax[1].bar(xs + w/2, df_sum["Ozgulluk"],  width=w, label='Ozgulluk',  color='#2ca02c')
ax[1].set_xticks(xs); ax[1].set_xticklabels([s.split()[0] for s in df_sum.index]); ax[1].set_ylim(0, 1.0); ax[1].set_title(f"Duyarlilik / Ozgulluk (esik={DECISION_THR})"); ax[1].legend()
plt.tight_layout(); plt.savefig("ext_summary.png", dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

def metrics_at(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn + 1e-9); spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9); f1 = 2 * prec * sens / (prec + sens + 1e-9)
    return {"Acc": (y_pred == y_true).mean(), "F1": f1, "Duyarlilik": sens, "Ozgulluk": spec,
            "Brier": brier_score_loss(y_true, y_prob), "ECE": expected_calibration_error(y_true, y_prob)}

CAL = {}
for name, r in RESULTS.items():
    yt, yp = r["y_true"], r["y_prob"]
    idx = np.arange(len(yt))
    cal_i, test_i = train_test_split(idx, test_size=0.5, stratify=yt, random_state=SEED)
    yt_c, yp_c = yt[cal_i], yp[cal_i]
    yt_t, yp_t = yt[test_i], yp[test_i]
    platt = LogisticRegression(C=1e6, solver='lbfgs').fit(yp_c.reshape(-1, 1), yt_c)
    yp_p  = platt.predict_proba(yp_t.reshape(-1, 1))[:, 1]
    iso   = IsotonicRegression(out_of_bounds='clip').fit(yp_c, yt_c)
    yp_i  = iso.transform(yp_t)
    fpr, tpr, _ = roc_curve(yt_t, yp_t); auc_t = auc(fpr, tpr)
    CAL[name] = {"yt": yt_t, "raw": yp_t, "platt": yp_p, "iso": yp_i, "auc": auc_t,
                 "m_raw": metrics_at(yt_t, yp_t), "m_platt": metrics_at(yt_t, yp_p),
                 "m_iso": metrics_at(yt_t, yp_i)}

rows = []
for name, c in CAL.items():
    for tag, key in [("Ham (0.5)", "m_raw"), ("Platt", "m_platt"), ("Isotonic", "m_iso")]:
        rows.append({"Set": name.split()[0], "Yontem": tag, **{k: round(v, 3) for k, v in c[key].items()}})
df_cal = pd.DataFrame(rows)
print("="*92)
print("  ESIK YENIDEN KALIBRASYONU  (gorulmemis %50 test yarisinda, esik=0.5)")
print("="*92)
print(df_cal.to_string(index=False))
print("="*92)
for name, c in CAL.items():
    print(f"  {name}: test AUC={c['auc']:.3f}  (kalibrasyon monoton -> AUC degismez)")

In [ ]:
n = len(CAL)
fig, axes = plt.subplots(n, 2, figsize=(13, 4.6 * n))
if n == 1:
    axes = np.array([axes])
fig.suptitle("Kalibrasyon: Öncesi vs Sonrası (görülmemiş test yarısı)", fontsize=13, fontweight='bold')

def rel_points(yt, yp, nb=10):
    bins = np.linspace(0, 1, nb + 1); idx = np.digitize(yp, bins) - 1
    bx, by = [], []
    for b in range(nb):
        m = idx == b
        if m.sum() > 0:
            bx.append(yp[m].mean()); by.append(yt[m].mean())
    return bx, by

for row, (name, c) in enumerate(CAL.items()):
    ax = axes[row][0]
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Ideal')
    for tag, key, col in [("Ham", "raw", "#999999"), ("Platt", "platt", "#1f77b4"), ("Isotonic", "iso", "#2ca02c")]:
        bx, by = rel_points(c["yt"], c[key])
        ax.plot(bx, by, 'o-', color=col, lw=1.8, ms=4, label=tag)
    ax.set_title(f"{name.split()[0]} — Güvenilirlik eğrisi"); ax.set_xlabel("Tahmin olasilik"); ax.set_ylabel("Gercek pnomoni orani")
    ax.legend(loc='upper left', fontsize=8)
    ax = axes[row][1]
    keys = ["m_raw", "m_platt", "m_iso"]; tags = ["Ham", "Platt", "Isotonic"]
    spec = [c[k]["Ozgulluk"] for k in keys]; sens = [c[k]["Duyarlilik"] for k in keys]
    x = np.arange(3); w = 0.35
    ax.bar(x - w/2, spec, w, label='Ozgulluk', color='#2ca02c')
    ax.bar(x + w/2, sens, w, label='Duyarlilik', color='#d62728')
    for i, (s, se) in enumerate(zip(spec, sens)):
        ax.text(i - w/2, s + 0.01, f"{s:.2f}", ha='center', fontsize=8)
        ax.text(i + w/2, se + 0.01, f"{se:.2f}", ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(tags); ax.set_ylim(0, 1.08)
    ax.set_title(f"{name.split()[0]} — Özgüllük / Duyarlılık (esik=0.5)"); ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig("ext_calibration.png", dpi=130, bbox_inches='tight'); plt.show()

## 11. Sonuçların Dürüst Yorumu <a id='11'></a>

### 11.1. AUC'yi nasıl okumalı?

- **AUC ≈ iç-test değeri (≈0.97):** Model dağıtım kaymasına dayanıklı; akciğer-odaklı mimarinin kısayolları gerçekten azalttığına dair güçlü kanıt.
- **AUC orta (≈0.80–0.90):** Kısmi genelleme. Pediatrik→yetişkin kayması ve etiket farkları göz önüne alındığında makul; ama klinik için yetersiz.
- **AUC düşük (≈0.5–0.7):** Model büyük ölçüde eğitim dağıtımına özgü ipuçlarına bağlı. İç-test skoru yanıltıcıydı; dağıtıma uzak.

### 11.2. Performans düşüşü *yalnızca* kötü model demek değildir

Bu dış setlerde bir düşüş beklenir ve birden çok kaynağı vardır; düşüşün ne kadarının gerçek model zayıflığı olduğunu ayrıştırmak önemlidir:

1. **Popülasyon kayması (en büyük etken):** Modeliniz 1–5 yaş pediatrik göğüs yapısında eğitildi; RSNA/NIH **yetişkindir**. Kalp/akciğer oranları, toraks şekli ve patoloji görünümü farklıdır. Hem ianpan segmentasyonu hem sınıflandırıcı bu kaymadan etkilenir.
2. **Etiket tanımı farkı:** RSNA'da pozitif sınıf "Lung Opacity"dir — pnömoni dışı opasiteleri de içerir. NIH'de "Pneumonia" etiketi rapordan NLP ile çıkarılmıştır (~%90 doğruluk). Yani "gerçek" etiketin kendisi gürültülüdür; bu, AUC'ye bir tavan koyar.
3. **Segmentasyon bağımlılığı:** Fallback oranı yüksekse (§8 çıktısı), ianpan yetişkin/farklı-cihaz görüntülerinde akciğeri tam bulamıyor olabilir; bu, sınıflandırıcıya bozuk girdi taşır. Fallback oranını mutlaka raporlayın.
4. **Kalibrasyon kayması:** Yüksek Brier/ECE, olasılıkların yeni dağıtımda güvenilmez olduğunu gösterir; AUC iyi olsa bile 0.5 eşiği optimal olmayabilir (oracle eşikle farka bakın).

### 11.3. Bu doğrulamanın kendi sınırları

Bu, **retrospektif, proxy-etiketli** bir dış doğrulamadır — güçlü bir gerçeklik kontrolüdür, ancak klinik/regülatuvar onay değildir. Tam doğrulama için prospektif veri, uzman radyolog mutabakat etiketleri ve alt-grup (yaş/cihaz/cinsiyet) analizi gerekir. Yine de "aynı dağıtımda %97" iddiasının ötesine geçen, tezinizi gerçekten sınayan adım budur.

### 11.4. Sonraki adımlar (düşüş büyükse)

- Yetişkin verisinden küçük bir alt kümeyle **alan-adaptasyonu / ince ayar** ve tekrar dış-test.
- Eşiği yeni sette **yeniden kalibre etme** (Platt/isotonic), olasılıkları düzeltme.
- Fallback/düşük-maske vakalarını ayrı raporlayıp ianpan'ın yetişkindeki davranışını denetleme.

> **Klinik Not.** Sonuç ne olursa olsun bu model bir karar-destek prototipidir; klinik kullanım için uzman denetimi, prospektif doğrulama ve regülasyon zorunludur.